In [ ]:
# M3 CELL 1 — source and label preflight (CPU is enough).
# Initial ETA ~10–90 s, depending on Google Drive speed; prints measured hash ETA.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path, PurePosixPath
from collections import Counter
from datetime import datetime, timezone
import csv, hashlib, json, time, zipfile

DATA_ROOT = Path('/content/drive/MyDrive/SentinelWard-data')
AUDIT_DIR = Path('/content/drive/MyDrive/SentinelWard-work/data-audit/20261015T132438_754996Z')
SOURCE = DATA_ROOT / 'challenge-2015-1.0.0' / 'reducing-false-arrhythmia-alarms-in-the-icu-the-physionetcomputing-in-cardiology-challenge-2015-1.0.0.zip'
LABELS = AUDIT_DIR / 'challenge2015_labels.csv'
VERIFY = AUDIT_DIR / 'nested_archive_verification.json'

assert SOURCE.is_file(), f'Missing source archive: {SOURCE}'
assert SOURCE.stat().st_size == 369541924, 'Source archive size changed; stop and audit it.'
assert LABELS.is_file() and VERIFY.is_file(), 'Run notebook 01 audit first.'
with LABELS.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == ['record_id', 'alarm_type', 'label']
    rows = list(reader)
assert len(rows) == 750
assert len({r['record_id'] for r in rows}) == 750
counts = Counter(r['label'] for r in rows)
assert counts == {'0': 456, '1': 294}, counts
prior = json.loads(VERIFY.read_text(encoding='utf-8'))['challenge-2015-1.0.0']
assert prior['nested_crc_pass'] and prior['label_header_waveform_match']
with zipfile.ZipFile(SOURCE) as archive:
    nested = [m for m in archive.infolist() if PurePosixPath(m.filename).name.lower() == 'training.zip']
    assert len(nested) == 1, 'Expected exactly one training.zip inside source archive.'
    nested_info = nested[0]

print('Hashing 2015 archive — estimated 10–90 s.', flush=True)
started = time.monotonic()
h = hashlib.sha256()
total = SOURCE.stat().st_size
done = 0
with SOURCE.open('rb') as f:
    for block in iter(lambda: f.read(32 * 1024**2), b''):
        h.update(block)
        done += len(block)
        elapsed = time.monotonic() - started
        eta = elapsed * (total - done) / done
        print(f'{done/total:.0%} | elapsed {elapsed:.1f}s | ETA {eta:.1f}s', flush=True)

report = {
    'source_archive': str(SOURCE), 'source_bytes': total,
    'source_sha256': h.hexdigest(),
    'nested_training_zip_bytes': nested_info.file_size,
    'audited_records': len(rows), 'alarm_labels': dict(counts),
    'prior_nested_crc_pass': True,
    'ecg_ppg_eligible_records': 'not yet verified',
    'patient_identity_for_split': 'not yet verified',
    'secalarmbench_generated': False,
    'm3_trained': False,
}
out_dir = Path('/content/drive/MyDrive/SentinelWard-work/m3')
out_dir.mkdir(parents=True, exist_ok=True)
out = out_dir / ('preflight_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.json')
out.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(); print('M3 PREFLIGHT PASS')
print(json.dumps(report, indent=2))
print('Saved:', out)
print(f'Completed in {time.monotonic()-started:.1f}s. No M3 training or test access.')


In [ ]:
# M3 CELL 2 — inventory paired ECG/PPG records; no waveform or test access.
# Initial ETA ~1–4 min on Drive, then a few seconds for 750 headers.
import shutil, re
from collections import Counter
started2 = time.monotonic()
local_zip = Path('/content/m3_training.zip')
with zipfile.ZipFile(SOURCE) as outer:
    info = next(m for m in outer.infolist() if PurePosixPath(m.filename).name.lower() == 'training.zip')
    if not local_zip.exists() or local_zip.stat().st_size != info.file_size:
        print('Staging nested training.zip locally — ETA depends on Drive speed.', flush=True)
        tmp = local_zip.with_suffix('.partial')
        copied = 0
        with outer.open(info) as src, tmp.open('wb') as dst:
            while chunk := src.read(16 * 1024**2):
                dst.write(chunk); copied += len(chunk)
                elapsed = max(time.monotonic()-started2, 0.001)
                print(f'{copied/info.file_size:.0%} | elapsed {elapsed:.1f}s | ETA {elapsed*(info.file_size-copied)/copied:.1f}s', flush=True)
        assert copied == info.file_size
        tmp.replace(local_zip)
print('Inspecting header inventory...', flush=True)
with zipfile.ZipFile(local_zip) as inner:
    assert inner.testzip() is None, 'Nested archive CRC failed.'
    headers = {PurePosixPath(n).stem: n for n in inner.namelist() if n.lower().endswith('.hea')}
    waveforms = {PurePosixPath(n).stem: n for n in inner.namelist() if n.lower().endswith('.mat')}
    assert len(headers) == len(waveforms) == len(rows) == 750, (len(headers), len(waveforms), len(rows))
    assert set(headers) == set(waveforms) == {r['record_id'] for r in rows}
    inventory = []
    for i, r in enumerate(rows, 1):
        rid = r['record_id']
        lines = inner.read(headers[rid]).decode('utf-8-sig').splitlines()
        parts = lines[0].split()
        assert parts[0] == rid, (rid, parts[0])
        n_sig = int(parts[1]); fs = float(parts[2]); n_samples = int(parts[3])
        signal_lines = [s for s in lines[1:] if s.strip() and not s.lstrip().startswith('#')][:n_sig]
        assert len(signal_lines) == n_sig, rid
        leads = [s.split()[-1].upper() for s in signal_lines]
        ecg = any(x in {'I','II','III','AVR','AVL','AVF','V','V1','V2','V3','V4','V5','V6','MCL1','MCL2'} for x in leads)
        ppg = 'PLETH' in leads
        inventory.append({'record_id': rid, 'alarm_type': r['alarm_type'], 'label': r['label'], 'fs_hz': fs, 'samples': n_samples, 'channels': '|'.join(leads), 'ecg_present': ecg, 'ppg_present': ppg, 'ecg_ppg_present': ecg and ppg})
        if i % 150 == 0: print(f'{i}/750 headers inspected', flush=True)
counts2 = Counter((x['ecg_present'], x['ppg_present']) for x in inventory)
paired = [x for x in inventory if x['ecg_ppg_present']]
report2 = {'records': len(inventory), 'paired_ecg_ppg': len(paired), 'channel_presence_counts': {str(k):v for k,v in counts2.items()}, 'paired_labels': dict(Counter(x['label'] for x in paired)), 'fs_hz': dict(Counter(x['fs_hz'] for x in inventory)), 'duration_seconds': dict(Counter(x['samples']/x['fs_hz'] for x in inventory)), 'patient_ids_available': False, 'signal_usability_verified': False, 'safe_patient_split_verified': False}
out_csv = out_dir / 'challenge2015_m3_header_inventory.csv'
with out_csv.open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(inventory[0])); w.writeheader(); w.writerows(inventory)
out_json = out_dir / 'challenge2015_m3_header_inventory_summary.json'
out_json.write_text(json.dumps(report2, indent=2), encoding='utf-8')
print('\nM3 HEADER INVENTORY')
print(json.dumps(report2, indent=2))
print('Saved:', out_csv, 'and', out_json)
print(f'Completed in {time.monotonic()-started2:.1f}s. No training, waveform decoding or split selection.')

In [ ]:
# M3 CELL 3 — verify paired waveforms in the 290–300 s pre-alarm window.
# Initial ETA ~20–120 s; progress reports measured ETA. No split or fitting.
import io
import numpy as np
from scipy.io import loadmat
assert local_zip.is_file() and len(paired) == 627, 'Run cells 1 and 2 first.'
t0 = time.monotonic()
checks = []
with zipfile.ZipFile(local_zip) as inner:
    for i, item in enumerate(paired, 1):
        rid = item['record_id']
        lines = inner.read(headers[rid]).decode('utf-8-sig').splitlines()
        head = lines[0].split(); nsig = int(head[1]); fs = float(head[2]); nsamp = int(head[3])
        lead_names = [s.split()[-1].upper() for s in lines[1:] if s.strip() and not s.lstrip().startswith('#')][:nsig]
        assert fs == 250 and len(lead_names) == nsig and nsamp >= 300*fs, rid
        raw = loadmat(io.BytesIO(inner.read(waveforms[rid])))['val']
        assert raw.shape == (nsig, nsamp), (rid, raw.shape, nsig, nsamp)
        segment = np.asarray(raw[:, int(290*fs):int(300*fs)], dtype=np.float64)
        assert segment.shape == (nsig, 2500), rid
        ecg_ids = [j for j, name in enumerate(lead_names) if name in {'I','II','III','AVR','AVL','AVF','V','V1','V2','V3','V4','V5','V6','MCL1','MCL2'}]
        ppg_ids = [j for j, name in enumerate(lead_names) if name == 'PLETH']
        assert ecg_ids and ppg_ids, rid
        def good(j):
            x = segment[j]
            return bool(np.isfinite(x).all() and np.unique(x).size > 1)
        ecg_good = sum(good(j) for j in ecg_ids)
        ppg_good = sum(good(j) for j in ppg_ids)
        checks.append({'record_id': rid, 'label': item['label'], 'alarm_type': item['alarm_type'], 'ecg_channels': len(ecg_ids), 'ppg_channels': len(ppg_ids), 'ecg_nonconstant_finite': ecg_good, 'ppg_nonconstant_finite': ppg_good, 'paired_basic_pass': bool(ecg_good and ppg_good), 'window_start_s': 290, 'window_end_s': 300})
        if i % 50 == 0 or i == len(paired):
            elapsed = time.monotonic()-t0
            print(f'{i}/{len(paired)} | elapsed {elapsed:.1f}s | ETA {elapsed*(len(paired)-i)/i:.1f}s', flush=True)
summary3 = {'paired_headers': len(paired), 'waveforms_shape_verified': len(checks), 'basic_prealarm_pair_pass': sum(x['paired_basic_pass'] for x in checks), 'basic_prealarm_pair_fail': sum(not x['paired_basic_pass'] for x in checks), 'failed_ids': [x['record_id'] for x in checks if not x['paired_basic_pass']], 'labels_among_basic_pass': dict(Counter(x['label'] for x in checks if x['paired_basic_pass'])), 'note': 'Finite and nonconstant checks are only basic technical checks; pulse/ECG signal quality, patient identity and leakage-safe split remain unverified.'}
csv3 = out_dir / 'challenge2015_m3_prealarm_waveform_checks.csv'
with csv3.open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(checks[0])); w.writeheader(); w.writerows(checks)
json3 = out_dir / 'challenge2015_m3_prealarm_waveform_summary.json'
json3.write_text(json.dumps(summary3, indent=2), encoding='utf-8')
print('M3 PRE-ALARM WAVEFORM CHECKS')
print(json.dumps(summary3, indent=2))
print('Saved:', csv3, 'and', json3)
print(f'Completed in {time.monotonic()-t0:.1f}s. No M3 training or test access.')

In [ ]:
# M3 CELL 4 — pin official generator code and load the 627 CinC ECG+PPG records.
# Initial ETA ~2–8 min. Local waveform copies vanish when this Colab runtime resets.
import os, sys, subprocess
from importlib.metadata import version
repo = Path('/content/IoMT_SecAlarmBench')
commit = '8b09a300c81348d16030c624900d27ee7d05e01f'
if not repo.exists():
    print('Cloning benchmark source — ETA ~10–60s.', flush=True)
    subprocess.run(['git','clone','--quiet','--depth','1','https://github.com/techosystem/IoMT_SecAlarmBench.git',str(repo)],check=True)
head = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
assert head == commit, f'Benchmark source changed: {head}; review before proceeding.'
try:
    import wfdb
except ModuleNotFoundError:
    print('Installing WFDB 4.1.2 — ETA ~10–60s.', flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--quiet','wfdb==4.1.2'],check=True)
    import wfdb
assert version('wfdb') == '4.1.2'
sys.path.insert(0,str(repo/'src'))
from cinc_loader import load_cinc_records
stage = Path('/content/m3_cinc_training'); stage.mkdir(exist_ok=True)
ids = sorted(x['record_id'] for x in paired)
t4 = time.monotonic()
with zipfile.ZipFile(local_zip) as inner:
    for i,rid in enumerate(ids,1):
        for name, lookup in (('.hea',headers),('.mat',waveforms)):
            dest = stage/(rid+name)
            if not dest.exists():
                with inner.open(lookup[rid]) as src, dest.open('wb') as dst:
                    shutil.copyfileobj(src,dst,1024*1024)
        if i % 100 == 0 or i == len(ids):
            elapsed = time.monotonic()-t4
            print(f'Extracted {i}/{len(ids)} records | elapsed {elapsed:.1f}s | ETA {elapsed*(len(ids)-i)/i:.1f}s',flush=True)
records = []
for start in range(0,len(ids),50):
    batch_ids = ids[start:start+50]
    batch_dir = Path('/content/m3_batch')
    batch_dir.mkdir(exist_ok=True)
    for link in batch_dir.iterdir(): link.unlink()
    for rid in batch_ids:
        for ext in ('.hea','.mat'):
            os.symlink(stage/(rid+ext),batch_dir/(rid+ext))
    records.extend(load_cinc_records(str(batch_dir),verbose=False))
    done = start+len(batch_ids); elapsed=time.monotonic()-t4
    print(f'Loaded {done}/{len(ids)} | elapsed {elapsed:.1f}s | ETA {elapsed*(len(ids)-done)/done:.1f}s',flush=True)
assert len(records)==627 and {r['record_id'] for r in records}==set(ids)
label_by_id={r['record_id']:r['label'] for r in rows}
assert all(int(r['true_alarm'])==int(label_by_id[r['record_id']]) for r in records)
source_audit={'repository':'https://github.com/techosystem/IoMT_SecAlarmBench','commit':head,'wfdb':version('wfdb'),'records_loaded':len(records),'true_alarm':sum(r['true_alarm'] for r in records),'false_alarm':sum(not r['true_alarm'] for r in records),'split_unit':'record_id only; actual patient identity unavailable','basic_290_300_second_pair_pass':594,'warning':'Record-disjoint is not proven patient-disjoint. Benchmark replay must be generated within splits.'}
source_report=out_dir/'m3_benchmark_source_audit.json'
source_report.write_text(json.dumps(source_audit,indent=2),encoding='utf-8')
print('SOURCE AND WAVEFORM LOAD PASS')
print(json.dumps(source_audit,indent=2))
print('Saved:',source_report)
print(f'Completed in {time.monotonic()-t4:.1f}s. No M3 training or test access.')

In [ ]:
# M3 CELL 5 — audit related s/l record IDs before any split.
# ETA ~1–10 s. This does not infer patient identity.
from collections import defaultdict
by_root=defaultdict(list)
for r in records:
    rid=r['record_id']
    assert rid[-1] in ('s','l') and rid[:-1], rid
    by_root[rid[:-1]].append(r)
paired_roots={k:v for k,v in by_root.items() if len(v)>1}
assert max(map(len,by_root.values()))<=2
examples=[]
for root,rs in list(sorted(paired_roots.items()))[:10]:
    examples.append({'root':root,'ids':[r['record_id'] for r in rs],'labels':[int(r['true_alarm']) for r in rs],'types':[r['alarm_type'] for r in rs]})
root_audit={'records':len(records),'base_id_groups':len(by_root),'groups_with_s_and_l':len(paired_roots),'single_record_groups':sum(len(v)==1 for v in by_root.values()),'groups_with_conflicting_labels':sum(len({r['true_alarm'] for r in v})>1 for v in by_root.values()),'groups_with_conflicting_alarm_types':sum(len({r['alarm_type'] for r in v})>1 for v in by_root.values()),'examples':examples,'split_rule':'Keep all records sharing the base ID in the same split; this is a conservative proxy, not a verified patient identifier.'}
root_file=out_dir/'m3_record_group_audit.json'
root_file.write_text(json.dumps(root_audit,indent=2),encoding='utf-8')
print('RECORD GROUP AUDIT')
print(json.dumps(root_audit,indent=2))
print('Saved:',root_file)

In [ ]:
# M3 CELL 6 — freeze a stratified, record-disjoint split before generating attacks.
# ETA ~2–15 s. This is NOT a verified patient-disjoint split.
from sklearn.model_selection import StratifiedShuffleSplit
import pandas as pd
start6=time.monotonic()
meta=pd.DataFrame([{'record_id':r['record_id'],'alarm_type':r['alarm_type'],'true_alarm':int(r['true_alarm'])} for r in records]).sort_values('record_id').reset_index(drop=True)
assert len(meta)==627 and meta.record_id.is_unique
# Exact duplicate waveform files cannot be separated across splits.
digests={}
for rid in meta.record_id:
    h=hashlib.sha256()
    with (stage/(rid+'.mat')).open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024**2),b''): h.update(chunk)
    digests[rid]=h.hexdigest()
assert len(set(digests.values()))==627, 'Exact duplicate waveform files found; group them before splitting.'
strata=meta.alarm_type.astype(str)+'|'+meta.true_alarm.astype(str)
outer_split=StratifiedShuffleSplit(n_splits=1,test_size=100,random_state=20261016)
dev_idx,test_idx=next(outer_split.split(meta,strata))
dev=meta.iloc[dev_idx].copy(); test=meta.iloc[test_idx].copy()
inner_split=StratifiedShuffleSplit(n_splits=1,test_size=91,random_state=20261017)
train_i,val_i=next(inner_split.split(dev,dev.alarm_type.astype(str)+'|'+dev.true_alarm.astype(str)))
train=dev.iloc[train_i].assign(split='train'); val=dev.iloc[val_i].assign(split='validation'); test=test.assign(split='test')
frozen=pd.concat([train,val,test],ignore_index=True).sort_values('record_id').reset_index(drop=True)
assert frozen.split.value_counts().to_dict()=={'train':436,'validation':91,'test':100}
assert frozen.record_id.is_unique and len(frozen)==627
split_path=out_dir/'m3_frozen_record_split_v1.csv'
serialized=frozen.to_csv(index=False)
if split_path.exists():
    assert split_path.read_text(encoding='utf-8')==serialized, 'Existing frozen split differs: STOP.'
else: split_path.write_text(serialized,encoding='utf-8')
split_hash=hashlib.sha256(serialized.encode()).hexdigest()
summary6={'counts':frozen.groupby('split').agg(records=('record_id','size'),true_alarms=('true_alarm','sum')).to_dict('index'),'waveform_sha256_duplicates':0,'record_overlap':0,'split_sha256':split_hash,'split_unit':'record ID; patient identity unverified','test_feature_extraction_started':False}
summary_path=out_dir/'m3_frozen_split_summary_v1.json'
summary_path.write_text(json.dumps(summary6,indent=2),encoding='utf-8')
print('FROZEN M3 SPLIT')
print(json.dumps(summary6,indent=2))
print('Saved:',split_path,summary_path)
print(f'Completed in {time.monotonic()-start6:.1f}s. No attack generation, fitting or test feature extraction.')

In [ ]:
# M3 CELL 7 — build only training and validation benchmark features.
# Initial ETA ~1–8 min on CPU; pilot gives a measured approximation.
import threading, gc
from injection_generator import GeneratorConfig
from eval_harness import build_dataset
assert split_path.is_file() and hashlib.sha256(split_path.read_bytes()).hexdigest()==split_hash
rec_by_id={r['record_id']:r for r in records}
train_ids=frozen.loc[frozen.split=='train','record_id'].tolist()
val_ids=frozen.loc[frozen.split=='validation','record_id'].tolist()
test_ids=set(frozen.loc[frozen.split=='test','record_id'])
assert not (set(train_ids)|set(val_ids))&test_ids
pilot_start=time.monotonic()
pilot_df,_=build_dataset([rec_by_id[x] for x in train_ids[:8]],GeneratorConfig(seed=20261016),verbose=False)
pilot_s=time.monotonic()-pilot_start
print(f'Pilot: {len(pilot_df)} rows from 8 train records in {pilot_s:.1f}s; rough total ETA {pilot_s*(len(train_ids)+len(val_ids))/8/60:.1f} min.',flush=True)
del pilot_df; gc.collect()
def make_part(name, ids, seed):
    subset=[rec_by_id[x] for x in ids]
    stop=threading.Event(); started=time.monotonic()
    def heartbeat():
        while not stop.wait(20):
            elapsed=time.monotonic()-started
            rough=max(0,pilot_s*len(ids)/8-elapsed)
            print(f'{name}: elapsed {elapsed/60:.1f} min | rough ETA {rough/60:.1f} min',flush=True)
    thread=threading.Thread(target=heartbeat,daemon=True); thread.start()
    try: table, cols=build_dataset(subset,GeneratorConfig(seed=seed),verbose=False)
    finally: stop.set(); thread.join(timeout=1)
    assert set(table.record_id)==set(ids)
    assert table.patient_id.astype(str).equals(table.record_id.astype(str)), 'Unexpected benchmark ID semantics.'
    table=table.drop(columns='patient_id').rename(columns={'record_id':'record_id'})
    assert table[cols].notna().all().all()
    path=out_dir/f'm3_{name}_benchmark_features_v1.csv.gz'
    table.to_csv(path,index=False,compression='gzip')
    print(f'{name}: {len(table)} windows | classes {table.cls.value_counts().to_dict()} | {len(cols)} features | elapsed {(time.monotonic()-started)/60:.1f} min',flush=True)
    return table,cols,path
train_df,feature_cols,train_path=make_part('train',train_ids,20261016)
val_df,val_cols,val_path=make_part('validation',val_ids,20261017)
assert feature_cols==val_cols
assert not (set(train_df.record_id)&set(val_df.record_id))
assert not (set(train_df.record_id)|set(val_df.record_id))&test_ids
feature_report={'train_rows':len(train_df),'validation_rows':len(val_df),'train_classes':train_df.cls.value_counts().to_dict(),'validation_classes':val_df.cls.value_counts().to_dict(),'feature_columns':feature_cols,'test_records_processed':0,'generator_commit':commit,'split_sha256':split_hash,'replay_boundary':'generation separately within each split'}
feature_report_path=out_dir/'m3_development_feature_report_v1.json'
feature_report_path.write_text(json.dumps(feature_report,indent=2),encoding='utf-8')
print('DEVELOPMENT FEATURE BUILD COMPLETE')
print(json.dumps({k:v for k,v in feature_report.items() if k!='feature_columns'},indent=2))
print('Saved:',train_path,val_path,feature_report_path)

In [ ]:
# M3 CELL 8 — five-seed Isolation Forest; fit on TRAIN normal windows only.
# Initial ETA ~20–120 s. Thresholds use VALIDATION normals at a 10% false-flag budget.
from google.colab import drive
drive.mount('/content/drive')
import json, time, hashlib, joblib
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score
root=Path('/content/drive/MyDrive/SentinelWard-work/m3')
report=json.loads((root/'m3_development_feature_report_v1.json').read_text())
cols=report['feature_columns']
train=pd.read_csv(root/'m3_train_benchmark_features_v1.csv.gz')
val=pd.read_csv(root/'m3_validation_benchmark_features_v1.csv.gz')
split_bytes=(root/'m3_frozen_record_split_v1.csv').read_bytes()
assert hashlib.sha256(split_bytes).hexdigest()==report['split_sha256']
split=pd.read_csv(root/'m3_frozen_record_split_v1.csv')
assert set(train.record_id)==set(split.loc[split.split=='train','record_id'])
assert set(val.record_id)==set(split.loc[split.split=='validation','record_id'])
assert not (set(train.record_id)|set(val.record_id))&set(split.loc[split.split=='test','record_id'])
Xtr=train.loc[train.cls=='physiology_normal',cols].to_numpy(dtype=np.float32)
Xval=val[cols].to_numpy(dtype=np.float32)
assert Xtr.shape==(22991,22) and Xval.shape==(11443,22)
assert np.isfinite(Xtr).all() and np.isfinite(Xval).all()
normal=(val.cls=='physiology_normal').to_numpy(); attack=(val.cls=='attack').to_numpy()
evalmask=normal|attack; y=attack[evalmask].astype(int)
run=root/('m3_iforest_'+time.strftime('%Y%m%dT%H%M%SZ',time.gmtime()))
run.mkdir(exist_ok=False)
results=[]; artifacts=[]; t8=time.monotonic()
for i,seed in enumerate(range(20261016,20261021),1):
    model=make_pipeline(StandardScaler(),IsolationForest(n_estimators=200,max_samples=256,contamination='auto',random_state=seed,n_jobs=2))
    model.fit(Xtr)
    score=-model.decision_function(Xval)
    threshold=float(np.quantile(score[normal],0.90,method='higher'))
    pred=score>threshold
    tn,fp,fn,tp=confusion_matrix(y,pred[evalmask],labels=[0,1]).ravel()
    metrics={'seed':seed,'threshold':threshold,'n_attack':int(attack.sum()),'n_normal':int(normal.sum()),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'attack_recall':float(tp/(tp+fn)),'normal_specificity':float(tn/(tn+fp)),'attack_precision':float(tp/(tp+fp)) if tp+fp else 0.0,'attack_auroc':float(roc_auc_score(y,score[evalmask])),'attack_average_precision':float(average_precision_score(y,score[evalmask])),'chance_ap':float(y.mean()),'fault_flag_rate':float(pred[(val.cls=='fault_artifact').to_numpy()].mean()),'physiology_event_flag_rate':float(pred[(val.cls=='physiology_event').to_numpy()].mean())}
    model_file=run/f'iforest_seed{seed}.joblib'; joblib.dump(model,model_file)
    pred_file=run/f'validation_predictions_seed{seed}.csv.gz'
    val[['record_id','window_id','cls','family','morphology']].assign(score=score,flag=pred.astype(int)).to_csv(pred_file,index=False,compression='gzip')
    model_hash=hashlib.sha256(model_file.read_bytes()).hexdigest()
    artifacts.append({'seed':seed,'model_file':model_file.name,'model_sha256':model_hash,'threshold':threshold})
    results.append(metrics)
    elapsed=time.monotonic()-t8
    print(f'Seed {i}/5 | AP {metrics["attack_average_precision"]:.4f} | AUROC {metrics["attack_auroc"]:.4f} | recall {metrics["attack_recall"]:.4f} | elapsed {elapsed:.1f}s | ETA {elapsed*(5-i)/i:.1f}s',flush=True)
metrics_df=pd.DataFrame(results)
metrics_df.to_csv(run/'validation_metrics.csv',index=False)
plan={'model':'M3 Isolation Forest benchmark baseline','benchmark_commit':report['generator_commit'],'split_sha256':report['split_sha256'],'feature_columns':cols,'train_normal_windows':len(Xtr),'validation_attack_windows':int(attack.sum()),'validation_normal_windows':int(normal.sum()),'threshold_rule':'validation physiology-normal 90th-percentile anomaly score; flag strictly above','seed_models':artifacts,'test_records_processed':0,'split_claim':'record-disjoint only; patient identity unavailable','replay_policy':'generated separately within split'}
(run/'frozen_test_plan.json').write_text(json.dumps(plan,indent=2),encoding='utf-8')
print('M3 VALIDATION RESULTS')
print(metrics_df.round(4).to_string(index=False))
print('MEAN AND SAMPLE SD')
print(metrics_df[['attack_recall','normal_specificity','attack_auroc','attack_average_precision','fault_flag_rate','physiology_event_flag_rate']].agg(['mean','std']).round(4).to_string())
print('Saved:',run)
print('Test records processed: 0')

In [ ]:
# M3 CELL 9 — audit detection by attack family and non-attack class.
# ETA ~2–10 s. Validation only; no threshold changes.
from pathlib import Path
import numpy as np, pandas as pd, json
from sklearn.metrics import average_precision_score, roc_auc_score
root=Path('/content/drive/MyDrive/SentinelWard-work/m3')
run=root/'m3_iforest_20261016T020922Z'
plan=json.loads((run/'frozen_test_plan.json').read_text())
val=pd.read_csv(root/'m3_validation_benchmark_features_v1.csv.gz',low_memory=False)
normal=(val.cls=='physiology_normal').to_numpy()
families=sorted(val.loc[val.cls=='attack','family'].dropna().unique())
rows9=[]
for entry in plan['seed_models']:
    pred=pd.read_csv(run/f'validation_predictions_seed{entry["seed"]}.csv.gz')
    assert len(pred)==len(val) and pred.record_id.astype(str).equals(val.record_id.astype(str))
    for family in families:
        positive=((val.cls=='attack')&(val.family==family)).to_numpy()
        keep=normal|positive
        y=positive[keep].astype(int); score=pred.score.to_numpy()[keep]
        rows9.append({'seed':entry['seed'],'scope':family,'n_positive':int(positive.sum()),'chance_ap':float(y.mean()),'ap':float(average_precision_score(y,score)),'auroc':float(roc_auc_score(y,score)),'flag_rate':float(pred.loc[positive,'flag'].mean())})
    for cls in ('fault_artifact','physiology_event'):
        mask=(val.cls==cls).to_numpy()
        rows9.append({'seed':entry['seed'],'scope':cls,'n_positive':int(mask.sum()),'chance_ap':float('nan'),'ap':float('nan'),'auroc':float('nan'),'flag_rate':float(pred.loc[mask,'flag'].mean())})
sub=pd.DataFrame(rows9)
sub.to_csv(run/'validation_subgroups.csv',index=False)
summary=sub.groupby('scope')[['n_positive','chance_ap','ap','auroc','flag_rate']].mean().round(4)
print('VALIDATION SUBGROUPS — mean across five seeds')
print(summary.to_string())
print('AP is compared with its own chance floor; faults and genuine events are reported as flag rates.')
print('Saved:',run/'validation_subgroups.csv')
print('Test records processed: 0')

In [ ]:
# M3 CELL 10 — prepare the held-out test features once, with frozen settings.
# Initial ETA ~1–3 min. No model fitting or threshold adjustment.
import json, hashlib, time, threading, gc
from pathlib import Path
import pandas as pd
from injection_generator import GeneratorConfig
from eval_harness import build_dataset
root=Path('/content/drive/MyDrive/SentinelWard-work/m3')
run=root/'m3_iforest_20261016T020922Z'; plan_file=run/'frozen_test_plan.json'
plan=json.loads(plan_file.read_text())
assert plan['test_records_processed']==0
plan.setdefault('test_generator_seed',20261018)
assert plan['test_generator_seed']==20261018
plan_file.write_text(json.dumps(plan,indent=2),encoding='utf-8')
plan_hash=hashlib.sha256(plan_file.read_bytes()).hexdigest()
for entry in plan['seed_models']:
    p=run/entry['model_file']
    assert hashlib.sha256(p.read_bytes()).hexdigest()==entry['model_sha256']
split_file=root/'m3_frozen_record_split_v1.csv'
assert hashlib.sha256(split_file.read_bytes()).hexdigest()==plan['split_sha256']
split=pd.read_csv(split_file)
test_ids=set(split.loc[split.split=='test','record_id'])
assert len(test_ids)==100 and 'records' in globals(), 'Restore source records using cell 4.'
test_records=[r for r in records if r['record_id'] in test_ids]
assert len(test_records)==100
path=root/'m3_test_benchmark_features_v1.csv.gz'
assert not path.exists(), 'Test features already prepared; do not regenerate them.'
print('Generating held-out test features from 100 records — ETA ~1–3 min.',flush=True)
t0=time.monotonic(); stop=threading.Event()
def heartbeat():
    while not stop.wait(20): print(f'Test preparation elapsed {(time.monotonic()-t0)/60:.1f} min | rough ETA {max(0,60-(time.monotonic()-t0))/60:.1f} min',flush=True)
thread=threading.Thread(target=heartbeat,daemon=True); thread.start()
try: test_df,test_cols=build_dataset(test_records,GeneratorConfig(seed=20261018),verbose=False)
finally: stop.set(); thread.join(timeout=1)
assert test_cols==plan['feature_columns']
assert set(test_df.record_id)==test_ids
assert not set(test_df.record_id)&set(split.loc[split.split!='test','record_id'])
assert test_df.patient_id.astype(str).equals(test_df.record_id.astype(str))
test_df=test_df.drop(columns='patient_id')
assert test_df[test_cols].notna().all().all()
test_df.to_csv(path,index=False,compression='gzip')
test_report={'test_records':100,'test_windows':len(test_df),'classes':test_df.cls.value_counts().to_dict(),'feature_count':len(test_cols),'split_sha256':plan['split_sha256'],'frozen_plan_sha256':plan_hash,'test_feature_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'model_scoring_performed':0,'replay_boundary':'test records only'}
(root/'m3_test_feature_report_v1.json').write_text(json.dumps(test_report,indent=2),encoding='utf-8')
print('FROZEN TEST FEATURES PREPARED')
print(json.dumps(test_report,indent=2))
print(f'Completed in {(time.monotonic()-t0)/60:.1f} min. No model scoring yet.')

In [ ]:
# M3 CELL 11 — single frozen test evaluation; do not tune from this output.
# ETA ~5–30 s. Five saved models and thresholds are read-only inputs.
from google.colab import drive
drive.mount('/content/drive')
import json, hashlib, time, joblib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score
root=Path('/content/drive/MyDrive/SentinelWard-work/m3')
run=root/'m3_iforest_20261016T020922Z'; plan_file=run/'frozen_test_plan.json'
plan_bytes=plan_file.read_bytes(); plan=json.loads(plan_bytes)
test_report=json.loads((root/'m3_test_feature_report_v1.json').read_text())
assert hashlib.sha256(plan_bytes).hexdigest()==test_report['frozen_plan_sha256']
assert hashlib.sha256((root/'m3_frozen_record_split_v1.csv').read_bytes()).hexdigest()==plan['split_sha256']
test_file=root/'m3_test_benchmark_features_v1.csv.gz'
assert hashlib.sha256(test_file.read_bytes()).hexdigest()==test_report['test_feature_sha256']
test=pd.read_csv(test_file,low_memory=False)
assert len(test)==12360 and set(test.record_id)==set(pd.read_csv(root/'m3_frozen_record_split_v1.csv').query("split == 'test'").record_id)
cols=plan['feature_columns']; X=test[cols].to_numpy(dtype=np.float32)
assert np.isfinite(X).all()
normal=(test.cls=='physiology_normal').to_numpy(); attack=(test.cls=='attack').to_numpy()
mask=normal|attack; y=attack[mask].astype(int)
result_dir=run/'frozen_test'; result_dir.mkdir(exist_ok=False)
results=[]; family_rows=[]; start=time.monotonic()
for i,entry in enumerate(plan['seed_models'],1):
    model_file=run/entry['model_file']
    assert hashlib.sha256(model_file.read_bytes()).hexdigest()==entry['model_sha256']
    model=joblib.load(model_file)
    score=-model.decision_function(X)
    flag=score>entry['threshold']
    tn,fp,fn,tp=confusion_matrix(y,flag[mask],labels=[0,1]).ravel()
    row={'seed':entry['seed'],'n_attack':int(attack.sum()),'n_normal':int(normal.sum()),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'attack_recall':float(tp/(tp+fn)),'normal_specificity':float(tn/(tn+fp)),'attack_precision':float(tp/(tp+fp)) if tp+fp else 0.0,'attack_auroc':float(roc_auc_score(y,score[mask])),'attack_average_precision':float(average_precision_score(y,score[mask])),'chance_ap':float(y.mean()),'fault_flag_rate':float(flag[(test.cls=='fault_artifact').to_numpy()].mean()),'physiology_event_flag_rate':float(flag[(test.cls=='physiology_event').to_numpy()].mean())}
    results.append(row)
    test[['record_id','window_id','cls','family','morphology']].assign(score=score,flag=flag.astype(int)).to_csv(result_dir/f'test_predictions_seed{entry["seed"]}.csv.gz',index=False,compression='gzip')
    for family in sorted(test.loc[attack,'family'].dropna().unique()):
        pos=((test.cls=='attack')&(test.family==family)).to_numpy(); keep=normal|pos
        family_rows.append({'seed':entry['seed'],'family':family,'n_attack':int(pos.sum()),'chance_ap':float(pos[keep].mean()),'ap':float(average_precision_score(pos[keep].astype(int),score[keep])),'auroc':float(roc_auc_score(pos[keep].astype(int),score[keep])),'flag_rate':float(flag[pos].mean())})
    elapsed=time.monotonic()-start
    print(f'Seed {i}/5 | AP {row["attack_average_precision"]:.4f} | recall {row["attack_recall"]:.4f} | elapsed {elapsed:.1f}s | ETA {elapsed*(5-i)/i:.1f}s',flush=True)
metrics=pd.DataFrame(results); families=pd.DataFrame(family_rows)
metrics.to_csv(result_dir/'test_metrics.csv',index=False)
families.to_csv(result_dir/'test_attack_families.csv',index=False)
(result_dir/'test_manifest.json').write_text(json.dumps({'frozen_plan_sha256':test_report['frozen_plan_sha256'],'test_feature_sha256':test_report['test_feature_sha256'],'split_sha256':plan['split_sha256'],'seeds':[x['seed'] for x in results],'test_tuning':False,'patient_disjoint_claim':False},indent=2),encoding='utf-8')
print('FROZEN M3 TEST RESULTS')
print(metrics.round(4).to_string(index=False))
print('MEAN AND SAMPLE SD')
print(metrics[['attack_recall','normal_specificity','attack_auroc','attack_average_precision','fault_flag_rate','physiology_event_flag_rate']].agg(['mean','std']).round(4).to_string())
print('ATTACK FAMILY MEAN')
print(families.groupby('family')[['n_attack','chance_ap','ap','auroc','flag_rate']].mean().round(4).to_string())
print('Saved:',result_dir)
print('No model fitting or threshold selection used test data.')

In [ ]:
# M3 CELL 12 — record-cluster uncertainty and honest completion card.
# ETA ~10–45 s. This cell only summarizes the already frozen test.
import time, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
root=Path('/content/drive/MyDrive/SentinelWard-work/m3')
run=root/'m3_iforest_20261016T020922Z'; result_dir=run/'frozen_test'
metrics=pd.read_csv(result_dir/'test_metrics.csv')
families=pd.read_csv(result_dir/'test_attack_families.csv')
pred=pd.read_csv(result_dir/'test_predictions_seed20261016.csv.gz',low_memory=False)
assert pred.record_id.nunique()==100
by_record=pred.groupby('record_id').indices
record_ids=np.array(sorted(by_record))
rng=np.random.default_rng(20261016); draws=[]; t0=time.monotonic()
for i in range(500):
    sampled=rng.choice(record_ids,size=len(record_ids),replace=True)
    idx=np.concatenate([by_record[r] for r in sampled])
    d=pred.iloc[idx]; keep=d.cls.isin(['physiology_normal','attack'])
    y=(d.loc[keep,'cls']=='attack').to_numpy(dtype=int)
    score=d.loc[keep,'score'].to_numpy(); flag=d.loc[keep,'flag'].to_numpy(dtype=int)
    if y.min()==y.max(): continue
    tp=((y==1)&(flag==1)).sum(); fn=((y==1)&(flag==0)).sum()
    tn=((y==0)&(flag==0)).sum(); fp=((y==0)&(flag==1)).sum()
    draws.append({'attack_recall':tp/(tp+fn),'normal_specificity':tn/(tn+fp),'attack_auroc':roc_auc_score(y,score),'attack_average_precision':average_precision_score(y,score)})
    if (i+1)%100==0:
        elapsed=time.monotonic()-t0
        print(f'Bootstrap {i+1}/500 | elapsed {elapsed:.1f}s | ETA {elapsed*(500-i-1)/(i+1):.1f}s',flush=True)
ci={k:{'low':float(np.percentile([d[k] for d in draws],2.5)),'high':float(np.percentile([d[k] for d in draws],97.5))} for k in draws[0]}
(result_dir/'record_cluster_bootstrap_seed20261016.json').write_text(json.dumps({'resamples':len(draws),'resample_unit':'record ID, not verified patient','seed':20261016,'percentile_95_interval':ci,'scope':'first frozen model only; descriptive uncertainty conditional on this corpus'},indent=2),encoding='utf-8')
mean=metrics[['attack_recall','normal_specificity','attack_auroc','attack_average_precision','fault_flag_rate','physiology_event_flag_rate']].mean()
fam=families.groupby('family')['flag_rate'].mean()
card=f'''# SentinelWard M3 — ECG/PPG anomaly baseline

Status: frozen benchmark baseline evaluated; full physiology-consistency specification remains open.

Source: IoMT-SecAlarmBench commit 8b09a300c81348d16030c624900d27ee7d05e01f, built from the audited CinC 2015 archive. Model: five 200-tree Isolation Forest fits on TRAIN physiology-normal windows only. Inputs: 22 published ECG, PPG and cross-correlation features. Threshold: validation normal-score 90th percentile, fixed before test.

Data: 627 ECG+PPG records; train/validation/test 436/91/100 records. The split is record-disjoint. Actual patient identity is unavailable, so patient-disjoint generalization is unverified. Attack replay sources were kept within each split.

Frozen test, mean across five seeds: attack AP {mean['attack_average_precision']:.4f} (chance {metrics.chance_ap.iloc[0]:.4f}); AUROC {mean['attack_auroc']:.4f}; attack recall {mean['attack_recall']:.4f}; normal specificity {mean['normal_specificity']:.4f}. Fault/artifact flag rate {mean['fault_flag_rate']:.4f}; genuine physiology-event flag rate {mean['physiology_event_flag_rate']:.4f}. Attack flag rate by family: FDI {fam['fdi']:.4f}, replay {fam['replay']:.4f}, spoofing {fam['spoofing']:.4f}.

Limits: replay detection is weak; faults are often flagged; the 22-feature baseline does not implement robust pulse-arrival-time, beat-rate agreement, or verified per-patient baselines in the original M3 plan. Synthetic PPG-only attacks and one source corpus limit transfer. The bootstrap interval is record-cluster descriptive uncertainty, not patient-level confidence. No clinical or security-deployment claim. Do not retune on the revealed test set.

Artifacts: frozen_test_plan.json, five joblib models, validation_metrics.csv, validation_subgroups.csv, frozen_test/test_metrics.csv, frozen_test/test_attack_families.csv, frozen_test/record_cluster_bootstrap_seed20261016.json.
'''
(run/'M3_MODEL_CARD.md').write_text(card,encoding='utf-8')
print('FIRST-SEED 95% RECORD-CLUSTER BOOTSTRAP INTERVALS')
print(json.dumps(ci,indent=2))
print('Saved:',run/'M3_MODEL_CARD.md')
print('Status: benchmark baseline complete; PAT/rate/per-patient M3 specification remains unfulfilled.')

In [ ]:
# M3 CELL 13 — pilot physiologic features on TRAIN records only.
# ETA ~10–40 s. This does not access the revealed test split.
import numpy as np, pandas as pd, time
from scipy.signal import find_peaks
from injection_generator import GeneratorConfig
from eval_harness import build_dataset
assert 'rec_by_id' in globals() and 'train_ids' in globals()
FS=125.0
new_cols=['ecg_rate_bpm','ppg_rate_bpm','rate_gap_bpm','beat_count_gap','ecg_rr_cv','ppg_rr_cv','pat_median_s','pat_iqr_s','pat_match_fraction','ecg_beats_valid','ppg_beats_valid']
def _beat_stats(signal, role):
    x=np.asarray(signal,dtype=np.float64)
    if not np.isfinite(x).all() or np.std(x)<0.05: return np.array([],dtype=int),0.0,0.0,x
    if role=='ecg':
        z=np.abs(x); peaks,_=find_peaks(z,distance=int(.30*FS),prominence=max(.35,.35*np.std(z)))
        return peaks,*_interval_stats(peaks),z
    choices=[]
    for sign in (1,-1):
        z=sign*x; peaks,props=find_peaks(z,distance=int(.33*FS),prominence=max(.25,.30*np.std(z)))
        rate,cv=_interval_stats(peaks)
        plausible=3<=len(peaks)<=30 and 30<=rate<=200
        quality=(1 if plausible else 0, -cv if plausible else -9, float(np.median(props['prominences'])) if len(peaks) else 0)
        choices.append((quality,peaks,rate,cv,z))
    _,peaks,rate,cv,z=max(choices,key=lambda a:a[0])
    return peaks,rate,cv,z
def _interval_stats(peaks):
    gaps=np.diff(peaks)/FS
    good=gaps[(gaps>=.30)&(gaps<=2.0)]
    if len(good)<2: return 0.0,0.0
    return float(60/np.median(good)),float(np.std(good)/(np.mean(good)+1e-9))
def physiology_features(window):
    ecg,ppg=np.asarray(window[:2],dtype=np.float64)
    r,hr_e,cv_e,_=_beat_stats(ecg,'ecg')
    p,hr_p,cv_p,oriented=_beat_stats(ppg,'ppg')
    feet=[]
    for peak in p:
        left=max(0,int(peak-.30*FS))
        if left<peak: feet.append(left+int(np.argmin(oriented[left:peak+1])))
    delays=[]
    for foot in feet:
        previous=r[r<foot]
        if len(previous):
            delay=(foot-previous[-1])/FS
            if .08<=delay<=.60: delays.append(delay)
    pat=np.array(delays,dtype=float)
    return [hr_e,hr_p,abs(hr_e-hr_p) if hr_e and hr_p else 0.0,abs(len(r)-len(p)),cv_e,cv_p,float(np.median(pat)) if len(pat) else 0.0,float(np.subtract(*np.percentile(pat,[75,25]))) if len(pat)>1 else 0.0,len(pat)/max(1,len(r)),int(hr_e>0),int(hr_p>0)]
t0=time.monotonic()
pilot_rows,_,pilot_windows=build_dataset([rec_by_id[x] for x in train_ids[:8]],GeneratorConfig(seed=20261016),return_windows=True,verbose=False)
extra=pd.DataFrame([physiology_features(w) for w in pilot_windows],columns=new_cols)
assert extra.shape==(len(pilot_rows),len(new_cols)) and np.isfinite(extra.to_numpy()).all()
normal=extra.loc[pilot_rows.cls=='physiology_normal']
print('TRAIN-ONLY FEATURE PILOT')
print('windows:',len(extra),'normal:',len(normal),'elapsed:',round(time.monotonic()-t0,1),'s')
print('ECG beat detection:',round(normal.ecg_beats_valid.mean(),3),'PPG beat detection:',round(normal.ppg_beats_valid.mean(),3))
print('PAT match fraction median:',round(normal.pat_match_fraction.median(),3),'PAT median among matched:',round(normal.loc[normal.pat_median_s>0,'pat_median_s'].median(),3))
print('Rate gap median:',round(normal.rate_gap_bpm.median(),1),'bpm')
print('No test data read or scored.')

In [ ]:
# M3 CELL 14 — extract beat-rate and pulse-arrival features for development only.
# Initial ETA ~5–9 min CPU. Outputs are resumable; old test data is never used.
import time, threading, gc, hashlib, json
import numpy as np, pandas as pd
from pathlib import Path
from injection_generator import GeneratorConfig
from eval_harness import build_dataset
root=Path('/content/drive/MyDrive/SentinelWard-work/m3')
base_report=json.loads((root/'m3_development_feature_report_v1.json').read_text())
old_cols=base_report['feature_columns']
assert len(records)==627 and len(train_ids)==436 and len(val_ids)==91
assert 'physiology_features' in globals() and len(new_cols)==11
def enrich_part(name,ids,seed):
    target=root/f'm3_{name}_physiology_features_v2.csv.gz'
    if target.exists():
        cached=pd.read_csv(target)
        assert len(cached)==base_report[f'{name}_rows'] and list(cached.columns)==['record_id','window_id']+new_cols
        print(f'{name}: verified cached physiology features, {len(cached)} rows.',flush=True)
        return target
    old=pd.read_csv(root/f'm3_{name}_benchmark_features_v1.csv.gz',low_memory=False)
    subset=[rec_by_id[x] for x in ids]
    started=time.monotonic(); stop=threading.Event()
    def heartbeat():
        while not stop.wait(25): print(f'{name}: regenerating aligned windows | elapsed {(time.monotonic()-started)/60:.1f} min | rough ETA {max(0, len(ids)*.55-(time.monotonic()-started))/60:.1f} min',flush=True)
    thread=threading.Thread(target=heartbeat,daemon=True); thread.start()
    try: fresh,cols,wins=build_dataset(subset,GeneratorConfig(seed=seed),return_windows=True,verbose=False)
    finally: stop.set(); thread.join(timeout=1)
    assert cols==old_cols and len(fresh)==len(old)==len(wins)
    for col in ['record_id','window_id','cls','family','morphology','level']:
        assert fresh[col].astype(str).equals(old[col].astype(str)),f'{name} row mismatch in {col}'
    np.testing.assert_allclose(fresh[old_cols].to_numpy(dtype=float),old[old_cols].to_numpy(dtype=float),rtol=1e-7,atol=1e-7)
    print(f'{name}: {len(wins)} windows match saved benchmark rows; extracting physiology features.',flush=True)
    values=[]; pstart=time.monotonic()
    for i,w in enumerate(wins,1):
        values.append(physiology_features(w))
        if i%5000==0 or i==len(wins):
            elapsed=time.monotonic()-pstart
            print(f'{name} physiology {i}/{len(wins)} | elapsed {elapsed:.1f}s | ETA {elapsed*(len(wins)-i)/i:.1f}s',flush=True)
    extra=pd.DataFrame(values,columns=new_cols)
    assert np.isfinite(extra.to_numpy(dtype=float)).all()
    result=pd.concat([old[['record_id','window_id']].reset_index(drop=True),extra],axis=1)
    result.to_csv(target,index=False,compression='gzip')
    normal=extra.loc[old.cls=='physiology_normal']
    print(f'{name}: saved {len(result)} rows | normal ECG beats valid {normal.ecg_beats_valid.mean():.3f} | PPG beats valid {normal.ppg_beats_valid.mean():.3f} | elapsed {(time.monotonic()-started)/60:.1f} min',flush=True)
    del fresh,wins,values,extra,result,old; gc.collect()
    return target
train_extra_path=enrich_part('train',train_ids,20261016)
val_extra_path=enrich_part('validation',val_ids,20261017)
manifest_v2={'feature_names':new_cols,'train_rows':base_report['train_rows'],'validation_rows':base_report['validation_rows'],'split_sha256':base_report['split_sha256'],'benchmark_commit':base_report['generator_commit'],'train_sha256':hashlib.sha256(train_extra_path.read_bytes()).hexdigest(),'validation_sha256':hashlib.sha256(val_extra_path.read_bytes()).hexdigest(),'test_features_touched_by_v2':False,'scope':'beat-rate and pulse-arrival features; no verified per-patient baseline'}
(root/'m3_physiology_features_v2_manifest.json').write_text(json.dumps(manifest_v2,indent=2),encoding='utf-8')
print('M3 V2 DEVELOPMENT FEATURES COMPLETE')
print(json.dumps(manifest_v2,indent=2))

In [ ]:
# M3 CELL 15 — fixed five-seed physiology-feature comparison (development only).
# ETA ~20–90 s. Same model, seeds and threshold rule as M3 baseline.
import time, json, hashlib, joblib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score
root=Path('/content/drive/MyDrive/SentinelWard-work/m3')
base=json.loads((root/'m3_development_feature_report_v1.json').read_text())
extra_manifest=json.loads((root/'m3_physiology_features_v2_manifest.json').read_text())
assert base['split_sha256']==extra_manifest['split_sha256']
assert extra_manifest['test_features_touched_by_v2'] is False
new_cols=extra_manifest['feature_names']; cols=base['feature_columns']+new_cols
parts={}
for name in ('train','validation'):
    p=root/f'm3_{name}_physiology_features_v2.csv.gz'
    assert hashlib.sha256(p.read_bytes()).hexdigest()==extra_manifest[f'{name}_sha256']
    old=pd.read_csv(root/f'm3_{name}_benchmark_features_v1.csv.gz',low_memory=False)
    fresh=pd.read_csv(p,low_memory=False)
    assert len(old)==len(fresh)==base[f'{name}_rows']
    for key in ('record_id','window_id'):
        assert old[key].astype(str).equals(fresh[key].astype(str)), f'{name}: {key} order mismatch'
    assert list(fresh.columns)==['record_id','window_id']+new_cols
    parts[name]=pd.concat([old.reset_index(drop=True),fresh[new_cols].reset_index(drop=True)],axis=1)
train2,val2=parts['train'],parts['validation']
assert not (set(train2.record_id)&set(val2.record_id))
Xtr=train2.loc[train2.cls=='physiology_normal',cols].to_numpy(np.float32)
Xva=val2[cols].to_numpy(np.float32)
assert Xtr.shape==(22991,33) and Xva.shape==(11443,33)
assert np.isfinite(Xtr).all() and np.isfinite(Xva).all()
normal=(val2.cls=='physiology_normal').to_numpy()
attack=(val2.cls=='attack').to_numpy()
mask=normal|attack; y=attack[mask].astype(int)
base_run=root/'m3_iforest_20261016T020922Z'
base_metrics=pd.read_csv(base_run/'validation_metrics.csv').set_index('seed')
base_families=pd.read_csv(base_run/'validation_subgroups.csv')
run2=root/('m3_v2_phys_iforest_'+time.strftime('%Y%m%dT%H%M%SZ',time.gmtime()))
run2.mkdir(exist_ok=False)
rows=[]; family_rows=[]; artifacts=[]; start=time.monotonic()
print('Comparing M3 v2 on validation only — ETA ~20–90 s.',flush=True)
for i,seed in enumerate(range(20261016,20261021),1):
    model=make_pipeline(StandardScaler(),IsolationForest(n_estimators=200,max_samples=256,contamination='auto',random_state=seed,n_jobs=2))
    model.fit(Xtr)
    score=-model.decision_function(Xva)
    threshold=float(np.quantile(score[normal],.90,method='higher'))
    flag=score>threshold
    tn,fp,fn,tp=confusion_matrix(y,flag[mask],labels=[0,1]).ravel()
    row={'seed':seed,'threshold':threshold,'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'attack_recall':float(tp/(tp+fn)),'normal_specificity':float(tn/(tn+fp)),'attack_precision':float(tp/(tp+fp)) if tp+fp else 0.,'attack_auroc':float(roc_auc_score(y,score[mask])),'attack_average_precision':float(average_precision_score(y,score[mask])),'fault_flag_rate':float(flag[(val2.cls=='fault_artifact').to_numpy()].mean()),'physiology_event_flag_rate':float(flag[(val2.cls=='physiology_event').to_numpy()].mean())}
    rows.append(row)
    for family in ('fdi','replay','spoofing'):
        fam=attack & (val2.family.to_numpy()==family)
        fam_mask=normal|fam
        family_rows.append({'seed':seed,'family':family,'n_attack':int(fam.sum()),'ap':float(average_precision_score(fam[fam_mask].astype(int),score[fam_mask])),'auroc':float(roc_auc_score(fam[fam_mask].astype(int),score[fam_mask])),'flag_rate':float(flag[fam].mean())})
    model_path=run2/f'iforest_seed{seed}.joblib'; joblib.dump(model,model_path)
    artifacts.append({'seed':seed,'model_file':model_path.name,'model_sha256':hashlib.sha256(model_path.read_bytes()).hexdigest(),'threshold':threshold})
    val2[['record_id','window_id','cls','family','morphology']].assign(score=score,flag=flag.astype(int)).to_csv(run2/f'validation_predictions_seed{seed}.csv.gz',index=False,compression='gzip')
    elapsed=time.monotonic()-start
    print(f'Seed {i}/5 | AP {row["attack_average_precision"]:.4f} | recall {row["attack_recall"]:.4f} | elapsed {elapsed:.1f}s | ETA {elapsed*(5-i)/i:.1f}s',flush=True)
metrics2=pd.DataFrame(rows); families2=pd.DataFrame(family_rows)
metrics2.to_csv(run2/'validation_metrics.csv',index=False)
families2.to_csv(run2/'validation_attack_families.csv',index=False)
keys=['attack_recall','normal_specificity','attack_auroc','attack_average_precision','fault_flag_rate','physiology_event_flag_rate']
compare=pd.DataFrame({'baseline_mean':base_metrics[keys].mean(),'v2_mean':metrics2[keys].mean()})
compare['v2_minus_baseline']=compare.v2_mean-compare.baseline_mean
compare.to_csv(run2/'comparison_to_baseline.csv')
plan={'model':'M3 v2 Isolation Forest with 11 fixed ECG/PPG timing features','feature_columns':cols,'split_sha256':base['split_sha256'],'benchmark_commit':base['generator_commit'],'threshold_rule':'validation normal 90th percentile; flag strictly above','seed_models':artifacts,'test_features_processed':0,'patient_disjoint_claim':False,'scope':'development validation; no independent v2 test'}
(run2/'validation_plan.json').write_text(json.dumps(plan,indent=2))
print('VALIDATION COMPARISON (positive delta is not always better):')
print(compare.round(4).to_string())
print('V2 ATTACK FAMILY FLAG RATES:')
print(families2.groupby('family')[['n_attack','ap','auroc','flag_rate']].mean().round(4).to_string())
print('BASELINE ATTACK FAMILY RESULTS:')
print(base_families.groupby('scope').mean(numeric_only=True).round(4).to_string())
print('Saved:',run2)
print('Test data accessed by M3 v2: 0')

In [ ]:
# M3 CELL 16 — one fixed supervised candidate on train records only.
# ETA ~20–120 s. This is an exploratory benchmark on synthetic attack labels.
import time, json, hashlib, joblib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score
assert 'train2' in globals() and 'val2' in globals() and 'base_metrics' in globals()
train_mask=train2.cls.isin(['physiology_normal','attack']).to_numpy()
X=train2.loc[train_mask,cols].to_numpy(np.float32)
ytr=(train2.loc[train_mask,'cls']=='attack').to_numpy(dtype=int)
assert len(X)==int(train_mask.sum()) and np.isfinite(X).all()
assert not (set(train2.record_id)&set(val2.record_id))
start=time.monotonic()
print('Training one fixed supervised M3 candidate — ETA ~20–120 s.',flush=True)
clf=HistGradientBoostingClassifier(max_iter=200,max_leaf_nodes=15,min_samples_leaf=50,learning_rate=0.05,l2_regularization=10.0,early_stopping=False,random_state=20261016)
clf.fit(X,ytr)
score=clf.predict_proba(Xva)[:,1]
threshold=float(np.quantile(score[normal],.90,method='higher'))
flag=score>threshold
tn,fp,fn,tp=confusion_matrix(y,flag[mask],labels=[0,1]).ravel()
row={'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'attack_recall':float(tp/(tp+fn)),'normal_specificity':float(tn/(tn+fp)),'attack_precision':float(tp/(tp+fp)) if tp+fp else 0.,'attack_auroc':float(roc_auc_score(y,score[mask])),'attack_average_precision':float(average_precision_score(y,score[mask])),'fault_flag_rate':float(flag[(val2.cls=='fault_artifact').to_numpy()].mean()),'physiology_event_flag_rate':float(flag[(val2.cls=='physiology_event').to_numpy()].mean())}
family_rows=[]
for family in ('fdi','replay','spoofing'):
    fam=attack & (val2.family.to_numpy()==family)
    keep=normal|fam
    family_rows.append({'family':family,'n_attack':int(fam.sum()),'ap':float(average_precision_score(fam[keep].astype(int),score[keep])),'auroc':float(roc_auc_score(fam[keep].astype(int),score[keep])),'flag_rate':float(flag[fam].mean())})
family_df=pd.DataFrame(family_rows)
base_family=base_families.groupby('scope')['flag_rate'].mean()
comparison=pd.DataFrame({'baseline_mean':base_metrics[list(row)].mean(),'supervised':pd.Series(row)})
comparison['delta']=comparison.supervised-comparison.baseline_mean
run3=root/('m3_v3_supervised_'+time.strftime('%Y%m%dT%H%M%SZ',time.gmtime()))
run3.mkdir(exist_ok=False)
model_path=run3/'hist_gradient_boosting.joblib'; joblib.dump(clf,model_path)
val2[['record_id','window_id','cls','family','morphology']].assign(score=score,flag=flag.astype(int)).to_csv(run3/'validation_predictions.csv.gz',index=False,compression='gzip')
pd.DataFrame([row]).assign(threshold=threshold).to_csv(run3/'validation_metrics.csv',index=False)
family_df.to_csv(run3/'validation_attack_families.csv',index=False)
comparison.to_csv(run3/'comparison_to_baseline.csv')
plan={'model':'M3 supervised HistGradientBoostingClassifier, exploratory synthetic-attack candidate','feature_columns':cols,'train_classes':train2.loc[train_mask,'cls'].value_counts().to_dict(),'split_sha256':base['split_sha256'],'benchmark_commit':base['generator_commit'],'threshold_rule':'validation normal 90th percentile; flag strictly above','threshold':threshold,'hyperparameters':clf.get_params(),'model_sha256':hashlib.sha256(model_path.read_bytes()).hexdigest(),'test_features_processed':0,'patient_disjoint_claim':False,'independent_accuracy_estimate':False}
(run3/'validation_plan.json').write_text(json.dumps(plan,indent=2,default=str))
print('VALIDATION: SUPERVISED VS BASELINE')
print(comparison.round(4).to_string())
print('ATTACK FAMILIES:')
print(family_df.round(4).to_string(index=False))
print('BASELINE FAMILY FLAG RATES:',base_family.round(4).to_dict())
print('Saved:',run3)
print(f'Elapsed {time.monotonic()-start:.1f}s | test data processed: 0')

In [ ]:
# M3 CELL 17 — fixed-feature ablation and record-level validation check.
# ETA ~15–60 s. No test data or new threshold tuning.
import time, json, hashlib, joblib
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score
assert 'train2' in globals() and 'val2' in globals() and 'run3' in globals()
t0=time.monotonic(); old_cols=base['feature_columns']
print('Checking whether timing features actually help — ETA ~15–60 s.',flush=True)
old_model=HistGradientBoostingClassifier(max_iter=200,max_leaf_nodes=15,min_samples_leaf=50,learning_rate=0.05,l2_regularization=10.0,early_stopping=False,random_state=20261016)
old_model.fit(train2.loc[train_mask,old_cols].to_numpy(np.float32),ytr)
old_score=old_model.predict_proba(val2[old_cols].to_numpy(np.float32))[:,1]
old_threshold=float(np.quantile(old_score[normal],.90,method='higher'))
old_flag=old_score>old_threshold
old_tn,old_fp,old_fn,old_tp=confusion_matrix(y,old_flag[mask],labels=[0,1]).ravel()
old_metrics={'features':22,'TN':int(old_tn),'FP':int(old_fp),'FN':int(old_fn),'TP':int(old_tp),'attack_recall':float(old_tp/(old_tp+old_fn)),'normal_specificity':float(old_tn/(old_tn+old_fp)),'attack_auroc':float(roc_auc_score(y,old_score[mask])),'attack_average_precision':float(average_precision_score(y,old_score[mask])),'fault_flag_rate':float(old_flag[(val2.cls=='fault_artifact').to_numpy()].mean()),'physiology_event_flag_rate':float(old_flag[(val2.cls=='physiology_event').to_numpy()].mean())}
full_metrics={'features':33,**{k:row[k] for k in old_metrics if k!='features'}}
abl=pd.DataFrame([old_metrics,full_metrics],index=['supervised_22','supervised_33'])
abl.to_csv(run3/'feature_ablation_validation.csv')
joblib.dump(old_model,run3/'hist_gradient_boosting_22features_ablation.joblib')
val_ids=np.asarray(val2.record_id.astype(str))
base_pred=pd.read_csv(base_run/'validation_predictions_seed20261016.csv.gz')
assert len(base_pred)==len(val2) and base_pred.record_id.astype(str).equals(val2.record_id.astype(str))
base_flag=base_pred.flag.to_numpy(dtype=bool)
attack_ids=np.unique(val_ids[attack]); rng=np.random.default_rng(20261016)
by_id={rid:np.flatnonzero(attack & (val_ids==rid)) for rid in attack_ids}
diffs=[]
for _ in range(500):
    picked=rng.choice(attack_ids,size=len(attack_ids),replace=True)
    idx=np.concatenate([by_id[rid] for rid in picked])
    diffs.append(float(flag[idx].mean()-base_flag[idx].mean()))
ci=np.quantile(diffs,[.025,.5,.975])
per_record=[]
for rid in attack_ids:
    idx=by_id[rid]
    per_record.append({'record_id':rid,'n_attack':len(idx),'supervised_attack_recall':float(flag[idx].mean()),'baseline_attack_recall':float(base_flag[idx].mean())})
per_record=pd.DataFrame(per_record)
per_record.to_csv(run3/'validation_per_record_attack_recall.csv',index=False)
summary={'ablation_metrics':abl.to_dict(orient='index'),'attack_records':len(attack_ids),'median_per_record_supervised_recall':float(per_record.supervised_attack_recall.median()),'median_per_record_baseline_recall':float(per_record.baseline_attack_recall.median()),'paired_record_bootstrap_recall_delta_95pct':[float(x) for x in ci],'bootstrap_replicates':500,'feature_columns_22':old_cols,'new_feature_columns':new_cols,'test_records_processed':0,'caveat':'Repeated windows within records; patient identity unknown; same synthetic attack generator.'}
(run3/'validation_robustness_summary.json').write_text(json.dumps(summary,indent=2))
print('SUPERVISED FEATURE ABLATION ON VALIDATION')
print(abl.round(4).to_string())
print('33-feature minus 22-feature AP:',round(abl.loc['supervised_33','attack_average_precision']-abl.loc['supervised_22','attack_average_precision'],4))
print('Attack records:',len(attack_ids),'| per-record median recall supervised:',round(per_record.supervised_attack_recall.median(),4),'baseline:',round(per_record.baseline_attack_recall.median(),4))
print('Paired record-bootstrap recall gain vs baseline, 95% interval:',np.round(ci,4))
print('Saved:',run3/'validation_robustness_summary.json')
print(f'Elapsed {time.monotonic()-t0:.1f}s | test data processed: 0')
# Preserve the simpler 22-feature candidate and its subgroup results.
family22=[]
for family in ('fdi','replay','spoofing'):
    fam=attack & (val2.family.to_numpy()==family)
    keep=normal|fam
    family22.append({'family':family,'n_attack':int(fam.sum()),'ap':float(average_precision_score(fam[keep].astype(int),old_score[keep])),'auroc':float(roc_auc_score(fam[keep].astype(int),old_score[keep])),'flag_rate':float(old_flag[fam].mean())})
family22=pd.DataFrame(family22)
family22.to_csv(run3/'validation_attack_families_22features.csv',index=False)
val2[['record_id','window_id','cls','family','morphology']].assign(score=old_score,flag=old_flag.astype(int)).to_csv(run3/'validation_predictions_22features.csv.gz',index=False,compression='gzip')
selected_path=run3/'hist_gradient_boosting_22features_ablation.joblib'
selected={'status':'preferred development candidate, not independently tested','selection_reason':'33 features add 0.0012 AP but raise fault-artifact flag rate from 0.1286 to 0.2221','model':'HistGradientBoostingClassifier','features':old_cols,'threshold':old_threshold,'threshold_rule':'90th percentile of validation normal scores; flag strictly above','model_sha256':hashlib.sha256(selected_path.read_bytes()).hexdigest(),'split_sha256':base['split_sha256'],'benchmark_commit':base['generator_commit'],'test_features_processed':0,'patient_disjoint_claim':False,'deployment_ready':False,'known_limitations':['synthetic attacks from same generator in train and validation','patient identities unavailable','validation influenced candidate choice','replay subgroup is small']}
(run3/'PREFERRED_DEVELOPMENT_MODEL.json').write_text(json.dumps(selected,indent=2))
print('PREFERRED 22-FEATURE ATTACK FAMILIES:')
print(family22.round(4).to_string(index=False))
print('Preferred candidate saved:',run3/'PREFERRED_DEVELOPMENT_MODEL.json')

In [ ]:
# M3 CELL 18 — restart-safe locked retrospective check; NEVER tune from this output.
# ETA ~5–30 s after Drive mounts. The baseline already revealed these test records.
from google.colab import drive
drive.mount('/content/drive')
import json, hashlib, joblib, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score
root=Path('/content/drive/MyDrive/SentinelWard-work/m3')
run3=root/'m3_v3_supervised_20261016T022833Z'
selected=json.loads((run3/'PREFERRED_DEVELOPMENT_MODEL.json').read_text())
cols=selected['features']; threshold=float(selected['threshold'])
assert selected['test_features_processed']==0 and len(cols)==22
model_file=run3/'hist_gradient_boosting_22features_ablation.joblib'
assert hashlib.sha256(model_file.read_bytes()).hexdigest()==selected['model_sha256']
model=joblib.load(model_file)
report_test=json.loads((root/'m3_test_feature_report_v1.json').read_text())
test_path=root/'m3_test_benchmark_features_v1.csv.gz'
assert hashlib.sha256(test_path.read_bytes()).hexdigest()==report_test['test_feature_sha256']
assert report_test['split_sha256']==selected['split_sha256']
test22=pd.read_csv(test_path,low_memory=False)
split=pd.read_csv(root/'m3_frozen_record_split_v1.csv')
test_ids=set(split.loc[split.split=='test','record_id']); dev_ids=set(split.loc[split.split!='test','record_id'])
assert set(test22.record_id)==test_ids and not (test_ids&dev_ids)
assert len(test22)==12360 and test22[cols].notna().all().all()
start=time.monotonic(); print('Scoring locked 22-feature model — ETA ~5–30 s.',flush=True)
test_score=model.predict_proba(test22[cols].to_numpy(np.float32))[:,1]
test_flag=test_score>threshold
tnormal=(test22.cls=='physiology_normal').to_numpy(); tattack=(test22.cls=='attack').to_numpy()
tkeep=tnormal|tattack; ty=tattack[tkeep].astype(int)
tn,fp,fn,tp=confusion_matrix(ty,test_flag[tkeep],labels=[0,1]).ravel()
result={'n_attack':int(tattack.sum()),'n_normal':int(tnormal.sum()),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'attack_recall':float(tp/(tp+fn)),'normal_specificity':float(tn/(tn+fp)),'attack_precision':float(tp/(tp+fp)) if tp+fp else 0.,'attack_auroc':float(roc_auc_score(ty,test_score[tkeep])),'attack_average_precision':float(average_precision_score(ty,test_score[tkeep])),'fault_flag_rate':float(test_flag[(test22.cls=='fault_artifact').to_numpy()].mean()),'physiology_event_flag_rate':float(test_flag[(test22.cls=='physiology_event').to_numpy()].mean())}
families=[]
for family in ('fdi','replay','spoofing'):
    fam=tattack & (test22.family.to_numpy()==family); keep=tnormal|fam
    families.append({'family':family,'n_attack':int(fam.sum()),'ap':float(average_precision_score(fam[keep].astype(int),test_score[keep])),'auroc':float(roc_auc_score(fam[keep].astype(int),test_score[keep])),'flag_rate':float(test_flag[fam].mean())})
check=run3/'retrospective_test_check'; check.mkdir(exist_ok=False)
pd.DataFrame([result]).to_csv(check/'metrics.csv',index=False)
pd.DataFrame(families).to_csv(check/'attack_families.csv',index=False)
test22[['record_id','window_id','cls','family','morphology']].assign(score=test_score,flag=test_flag.astype(int)).to_csv(check/'predictions.csv.gz',index=False,compression='gzip')
(check/'provenance.json').write_text(json.dumps({'model_sha256':selected['model_sha256'],'threshold':threshold,'feature_sha256':report_test['test_feature_sha256'],'split_sha256':selected['split_sha256'],'model_or_threshold_tuned_after_test':False,'test_was_previously_revealed':True,'fresh_independent_test':False},indent=2))
base_test=pd.read_csv(root/'m3_iforest_20261016T020922Z'/'frozen_test'/'test_metrics.csv')
print('LOCKED RETROSPECTIVE TEST — baseline mean vs preferred supervised model')
print(pd.DataFrame({'baseline_mean':base_test[list(result)].mean(numeric_only=True),'supervised_22':pd.Series(result)}).round(4).to_string())
print('ATTACK FAMILIES:')
print(pd.DataFrame(families).round(4).to_string(index=False))
print('Saved:',check)
print(f'Elapsed {time.monotonic()-start:.1f}s. No refitting or threshold change.')